In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from tkinter import Tk, Button, Frame, Label
import tkinter.font as tkFont

# ========== 網頁爬蟲部分 ==========
def scrape_articles():
    print("🔍 開始爬取 NBA 資訊...")
    driver = webdriver.Chrome()
    url = 'https://sports.sina.com.cn/nba/?from=wap'
    driver.get(url)

    try:
        element_present = EC.presence_of_element_located((By.CLASS_NAME, 'feed-card-item'))
        WebDriverWait(driver, 10).until(element_present)
    except TimeoutException:
        print("⚠️ Timed out waiting for page to load")

    titles, times, links = [], [], []

    for _ in range(3):  # 只抓前幾頁以避免被封
        sleep(2)
        articles = driver.find_elements(By.CLASS_NAME, 'feed-card-item')

        for article in articles:
            try:
                title = article.find_element(By.TAG_NAME, 'a').text
                href = article.find_element(By.TAG_NAME, 'a').get_attribute('href')
                soup = BeautifulSoup(article.get_attribute('innerHTML'), 'html.parser')
                time_tag = soup.find('div', class_='feed-card-time')
                time_text = time_tag.text.strip() if time_tag else "N/A"
                titles.append(title)
                times.append(time_text)
                links.append(href)
            except:
                continue

        try:
            next_page_url = driver.find_element(By.CSS_SELECTOR, '.pagination-next').get_attribute('href')
            driver.get(next_page_url)
        except:
            break

    driver.quit()
    print(f"✅ 共擷取 {len(titles)} 筆文章資料。")
    return titles, times, links

# ========== 分析資料與作圖 ==========
def analyze_and_plot(titles):
    players_chinese = ['约基奇', '莫兰特', '哈利伯顿', '杜兰特', '塔图姆', '亚历山大', '恩比德', '詹姆斯', '库里']
    players_english = ['Jokic', 'Morant', 'Haliburton', 'Durant', 'Tatum', 'SGA', 'Embiid', 'James', 'Curry']
    colors = ['#427f8f', '#4a8fa1', '#559db0', '#66a7b8', '#77b1c0', '#89bbc8', '#9ac5d0', '#bdd9e0', '#cee3e8']

    counts = []
    for name in players_chinese:
        count = sum(name in title for title in titles)
        counts.append(count)

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(players_english, counts, color=colors)
    ax.set_xlabel('Player', fontsize=14)
    ax.set_ylabel('Mention Count', fontsize=14)
    ax.set_title('📊 NBA Player Volume Analysis', fontsize=18)
    ax.set_xticklabels(players_english, rotation=45, ha='right')

    # 顯示數值
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count),
                ha='center', va='bottom', fontsize=10)

    return fig

# ========== 建立 GUI ==========
def plot_graph():
    titles, _, _ = scrape_articles()
    fig = analyze_and_plot(titles)

    for widget in plot_frame.winfo_children():
        widget.destroy()

    canvas = FigureCanvasTkAgg(fig, master=plot_frame)
    canvas.draw()
    canvas.get_tk_widget().pack()

# 主視窗設定
root = Tk()
root.geometry('820x850')
root.title(" NBA 熱門球員文章分析")

# 字體樣式
header_font = tkFont.Font(family="Helvetica", size=18, weight="bold")

# 頁首標題
header_label = Label(root, text="NBA 熱門球員文章分析", font=header_font, pady=20)
header_label.pack()

# 繪圖區域
plot_frame = Frame(root)
plot_frame.pack(pady=10)

# 按鈕
plot_button = Button(root, text="點擊爬取並分析文章", command=plot_graph,
                     font=('Helvetica', 14), width=30, height=2, bg="#3182bd", fg="white")
plot_button.pack(pady=20)

# 開始事件迴圈
root.mainloop()


🔍 開始爬取 NBA 資訊...
✅ 共擷取 30 筆文章資料。


/var/folders/mf/yg3qhjxs7rv2j51ph7b70pc00000gn/T/ipykernel_3140/75121893.py:71: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(players_english, rotation=45, ha='right')
/var/folders/mf/yg3qhjxs7rv2j51ph7b70pc00000gn/T/ipykernel_3140/75121893.py:89: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  canvas.draw()
/Users/alan/anaconda3/lib/python3.11/tkinter/__init__.py:861: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  func(*args)
